In [ ]:
# Parameters
city_key = "ywg"
analysis_feed_id = "current"
save_figures = False
figures_dir = "reports/pr2/figures"
dpi = 200

# PR2: Capacity and Reliability Priorities

This notebook ranks routes by service stress proxies and summarizes reliability by PTN tier. It uses canonical analyzer outputs instead of notebook-local SQL.

## 1. Setup & Helpers

In [ ]:
from pathlib import Path
import warnings

import contextily as cx
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from shapely.geometry import LineString

warnings.filterwarnings("ignore")

from ptn_analysis.context.reporting import (
    ensure_report_dirs,
    save_placeholder_figure,
    save_report_figure,
)

ensure_report_dirs("pr2")
figure_output_directory = Path(figures_dir)
figure_output_directory.mkdir(parents=True, exist_ok=True)

from ptn_analysis import TransitContext
from ptn_analysis.analysis import PTN_TIER_COLORS, PTN_TIER_ORDER


def build_route_gdf(db, feed_id="current"):
    """Build a GeoDataFrame of route geometries from GTFS shapes."""
    shape_routes = db.query(
        """
        SELECT r.route_short_name, t.shape_id, COUNT(*) as trip_count
        FROM ywg_trips t
        JOIN ywg_routes r ON t.route_id = r.route_id AND t.feed_id = r.feed_id
        WHERE t.feed_id = :feed_id AND t.shape_id IS NOT NULL
        GROUP BY r.route_short_name, t.shape_id
        ORDER BY r.route_short_name, trip_count DESC
        """,
        {"feed_id": feed_id},
    )
    best_shapes = shape_routes.drop_duplicates(subset="route_short_name", keep="first")
    shape_ids = best_shapes["shape_id"].unique().tolist()
    shapes = db.query(
        f"""
        SELECT shape_id, shape_pt_lat, shape_pt_lon, shape_pt_sequence
        FROM ywg_shapes
        WHERE shape_id IN ({','.join(f"'{s}'" for s in shape_ids)})
        ORDER BY shape_id, shape_pt_sequence
        """
    )
    lines = {}
    for shape_id, group in shapes.groupby("shape_id"):
        coords = list(zip(group["shape_pt_lon"], group["shape_pt_lat"]))
        if len(coords) >= 2:
            lines[shape_id] = LineString(coords)
    best_shapes = best_shapes.copy()
    best_shapes["geometry"] = best_shapes["shape_id"].map(lines)
    best_shapes = best_shapes.dropna(subset=["geometry"])
    return gpd.GeoDataFrame(best_shapes, geometry="geometry", crs="EPSG:4326")

## 2. Load Capacity and Reliability Tables

In [ ]:
ctx = TransitContext.from_defaults(feed_id=analysis_feed_id)
frequency_analyzer = ctx.frequency()
coverage_analyzer = ctx.coverage()
capacity_priority_table = frequency_analyzer.build_capacity_priority_table(top_n=25)
reliability_table = frequency_analyzer.build_route_reliability_table()
route_fact_table = frequency_analyzer.build_route_schedule_fact_table()
priority_matrix_table = coverage_analyzer.build_priority_metrics_table()

print(f"Capacity rows: {len(capacity_priority_table)}")
print(f"Reliability rows: {len(reliability_table)}")
print(f"Priority matrix rows: {len(priority_matrix_table)}")
display(capacity_priority_table.head())

## 3. Capacity Stress Scatter

In [ ]:
# Compute capacity data for combined figure (plot deferred to combined cell)
if not capacity_priority_table.empty:
    cap_route_gdf = build_route_gdf(ctx.working_db, feed_id=analysis_feed_id)
    cap_map_df = cap_route_gdf.merge(
        capacity_priority_table[["route_short_name", "upgrade_priority_score",
                                  "passups_per_100k_boardings", "weekday_boardings"]],
        on="route_short_name", how="inner",
    ).to_crs(epsg=3857)
    cap_neigh_gdf = ctx.working_db.neighbourhood_gdf().to_crs(epsg=3857)
    cap_map_df["line_width"] = 1.5 + 3 * (
        cap_map_df["upgrade_priority_score"] / cap_map_df["upgrade_priority_score"].max()
    )
    print(f"Capacity data ready: {len(cap_map_df)} routes")
else:
    cap_map_df = None
    cap_neigh_gdf = None



## 4. Upgrade Priority Ranking

In [ ]:
def _best_label_point(geom, placed, candidates=None):
    """Pick the point along a LineString/MultiLineString farthest from already-placed labels."""
    from shapely.ops import nearest_points
    from shapely.geometry import Point, MultiPoint
    if candidates is None:
        candidates = [0.15, 0.25, 0.35, 0.5, 0.65, 0.75, 0.85]
    pts = []
    for f in candidates:
        try:
            pts.append(geom.interpolate(f, normalized=True))
        except Exception:
            pass
    if not pts:
        return geom.centroid
    if not placed:
        # First label: pick 0.35 (slightly off-center, away from downtown convergence)
        return pts[2] if len(pts) > 2 else pts[0]
    placed_mp = MultiPoint(placed)
    best_pt, best_dist = pts[0], 0
    for pt in pts:
        d = pt.distance(placed_mp)
        if d > best_dist:
            best_dist = d
            best_pt = pt
    return best_pt

if capacity_priority_table.empty:
    save_placeholder_figure("upgrade_priority.png", "Capacity priority data is missing.", figures_dir=figure_output_directory, dpi=dpi, enabled=save_figures)
else:
    top_priority_table = capacity_priority_table.head(10).copy()
    route_gdf = build_route_gdf(ctx.working_db, feed_id=analysis_feed_id)

    all_map = route_gdf.to_crs(epsg=3857)
    top_map = route_gdf.merge(
        top_priority_table[["route_short_name", "upgrade_priority_score", "ptn_tier", "recommendation"]],
        on="route_short_name", how="inner",
    ).to_crs(epsg=3857)
    neigh_gdf = ctx.working_db.neighbourhood_gdf().to_crs(epsg=3857)

    fig, ax = plt.subplots(figsize=(10, 8))
    neigh_gdf.plot(ax=ax, facecolor="none", edgecolor="#cccccc", linewidth=0.3)
    all_map.plot(ax=ax, color="#d9d9d9", linewidth=0.8, alpha=0.5)
    top_sorted = top_map.sort_values("upgrade_priority_score", ascending=True)
    top_sorted.plot(
        column="upgrade_priority_score", ax=ax, legend=True,
        cmap="YlOrRd", linewidth=3.5,
        legend_kwds={"label": "Upgrade Priority Score", "shrink": 0.6},
    )
    placed = []
    for _, row in top_map.sort_values("upgrade_priority_score", ascending=False).iterrows():
        pt = _best_label_point(row.geometry, placed)
        placed.append(pt)
        score = row["upgrade_priority_score"]
        tier = row.get("ptn_tier", "")
        ax.annotate(
            f"{row['route_short_name']} ({tier})\n{score:.0f}",
            (pt.x, pt.y),
            fontsize=8, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor="gray", alpha=0.85),
        )
    cx.add_basemap(ax, source=cx.providers.CartoDB.Positron, zoom=12)
    ax.set_title("Top 10 Upgrade Priority Routes", fontsize=13, fontweight="bold")
    ax.set_axis_off()
    plt.tight_layout()
    save_report_figure(fig, "upgrade_priority.png", figures_dir=figure_output_directory, dpi=dpi, enabled=save_figures)
    plt.close(fig)
    display(top_priority_table[["route_short_name", "ptn_tier", "upgrade_priority_score", "recommendation"]])


## 5. Reliability by PTN Tier

In [ ]:
def _best_label_point(geom, placed, candidates=None):
    """Pick the point along a LineString/MultiLineString farthest from already-placed labels."""
    from shapely.ops import nearest_points
    from shapely.geometry import Point, MultiPoint
    if candidates is None:
        candidates = [0.15, 0.25, 0.35, 0.5, 0.65, 0.75, 0.85]
    pts = []
    for f in candidates:
        try:
            pts.append(geom.interpolate(f, normalized=True))
        except Exception:
            pass
    if not pts:
        return geom.centroid
    if not placed:
        # First label: pick 0.35 (slightly off-center, away from downtown convergence)
        return pts[2] if len(pts) > 2 else pts[0]
    placed_mp = MultiPoint(placed)
    best_pt, best_dist = pts[0], 0
    for pt in pts:
        d = pt.distance(placed_mp)
        if d > best_dist:
            best_dist = d
            best_pt = pt
    return best_pt

if reliability_table.empty:
    save_placeholder_figure("reliability_ontime.png", "Reliability data is missing.",
                            figures_dir=figure_output_directory, dpi=dpi, enabled=save_figures)
else:
    route_gdf = build_route_gdf(ctx.working_db, feed_id=analysis_feed_id)
    map_df = route_gdf.merge(
        reliability_table[["route_short_name", "pct_on_time", "ptn_tier", "mean_deviation_sec"]],
        on="route_short_name", how="inner",
    ).to_crs(epsg=3857)
    neigh_gdf = ctx.working_db.neighbourhood_gdf().to_crs(epsg=3857)

    fig, ax = plt.subplots(figsize=(10, 8))
    neigh_gdf.plot(ax=ax, facecolor="#f7f7f7", edgecolor="#cccccc", linewidth=0.3)
    map_sorted = map_df.sort_values("pct_on_time", ascending=False)
    map_sorted.plot(
        column="pct_on_time", ax=ax, legend=True,
        cmap="RdYlGn", linewidth=3.0, alpha=0.85,
        legend_kwds={"label": "On-time %", "shrink": 0.5},
    )
    worst = map_df.nsmallest(8, "pct_on_time")
    placed = []
    for _, row in worst.iterrows():
        pt = _best_label_point(row.geometry, placed)
        placed.append(pt)
        delay_sec = row.get("mean_deviation_sec", 0)
        delay_str = f"  +{delay_sec/60:.0f}m late" if delay_sec and delay_sec > 0 else ""
        ax.annotate(f"{row['route_short_name']} ({row.get('ptn_tier','')})\n{row['pct_on_time']:.0f}% on-time{delay_str}",
                    (pt.x, pt.y), fontsize=6.5, fontweight="bold", ha="center",
                    bbox=dict(boxstyle="round,pad=0.15", facecolor="white",
                              edgecolor="gray", alpha=0.9))
    route_bounds = map_df.total_bounds
    x_pad = (route_bounds[2] - route_bounds[0]) * 0.08
    y_pad = (route_bounds[3] - route_bounds[1]) * 0.08
    ax.set_xlim(route_bounds[0] - x_pad, route_bounds[2] + x_pad)
    ax.set_ylim(route_bounds[1] - y_pad, route_bounds[3] + y_pad)
    cx.add_basemap(ax, source=cx.providers.CartoDB.Positron, zoom=12)
    ax.set_title("On-time Performance by Route", fontsize=13, fontweight="bold")
    ax.set_axis_off()
    plt.tight_layout()
    save_report_figure(fig, "reliability_ontime.png",
                       figures_dir=figure_output_directory, dpi=dpi, enabled=save_figures)
    plt.close(fig)


## 7. Interpretation

Use the exported capacity table as the common input for Ahmed's synthesis, Stephenie's summary panel, and the final report recommendations.

## 6. Jobs-adjusted Priority Matrix

This table combines employment access, income context, and transit demand stress to prioritise upgrade candidates.

In [ ]:
if priority_matrix_table.empty:
    print("Priority matrix data is missing.")
else:
    sort_column = "priority_rank" if "priority_rank" in priority_matrix_table.columns else "priority_score"
    display(
        priority_matrix_table.sort_values(
            sort_column,
            ascending=True,
        ).head(15)
    )

## 8. Passup Risk Model — Negative Binomial GLM

Passup counts are overdispersed non-negative integers → Negative Binomial regression is preferred over Poisson. The trained model scores each route's capacity upgrade urgency based on its passup rate, headway, and on-time performance.

In [ ]:
import numpy as np
import statsmodels.api as sm
from joblib import dump
from ptn_analysis.analysis.frequency import MODELS_DIR

ctx_full = TransitContext.from_defaults(feed_id=analysis_feed_id)
fa_full = ctx_full.frequency()
all_routes_capacity_df = fa_full.calculate_capacity_stress(top_n=9999)

if all_routes_capacity_df.empty:
    print("Capacity data not available. Run make data first.")
else:
    all_routes_capacity_df["log_headway"] = np.log1p(
        all_routes_capacity_df["mean_headway_minutes"].fillna(30)
    )
    all_routes_capacity_df["log_boardings"] = np.log1p(
        all_routes_capacity_df["weekday_boardings"].fillna(0)
    )
    all_routes_capacity_df["pct_on_time_filled"] = all_routes_capacity_df["pct_on_time"].fillna(0)

    endog = all_routes_capacity_df["total_passups"].fillna(0).astype(int)
    exog = sm.add_constant(
        all_routes_capacity_df[["log_headway", "log_boardings", "pct_on_time_filled"]].fillna(0)
    )

    nb_model = sm.GLM(endog, exog, family=sm.families.NegativeBinomial(alpha=1.0)).fit()
    poisson_model = sm.GLM(endog, exog, family=sm.families.Poisson()).fit()
    print(nb_model.summary())
    print(f"\nAIC — NegBin: {nb_model.aic:.1f}  |  Poisson: {poisson_model.aic:.1f}")
    print("Lower AIC for NegBin confirms overdispersion (count variance exceeds mean).")

    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    dump(nb_model, MODELS_DIR / "passup_nb_v1.pkl")
    print(f"Model saved → {MODELS_DIR / 'passup_nb_v1.pkl'}")

    all_routes_capacity_df["passup_risk_score"] = nb_model.predict(exog)

    WINNIPEG_ADULT_FARE = 3.15
    all_routes_capacity_df["lost_revenue_est"] = (
        all_routes_capacity_df["total_passups"].fillna(0) * WINNIPEG_ADULT_FARE
    ).round(2)

    q50 = all_routes_capacity_df["passup_risk_score"].quantile(0.50)
    q85 = all_routes_capacity_df["passup_risk_score"].quantile(0.85)
    all_routes_capacity_df["recommendation"] = all_routes_capacity_df["passup_risk_score"].apply(
        lambda v: "LRT Candidate" if v >= q85 else ("BRT Candidate" if v >= q50 else "Bus Ops Tuning")
    )

    routes_with_passup_risk = all_routes_capacity_df.sort_values(
        "passup_risk_score",
        ascending=False,
    )[[
        "route_short_name", "ptn_tier", "total_passups", "passup_risk_score",
        "lost_revenue_est", "recommendation"
    ]]
    display(routes_with_passup_risk.head(15))

## 9. Transit-Oriented Development Gap Analysis

**Original finding:** High-density residential neighbourhoods may be close to a frequent bus corridor (e.g., Portage Ave, Henderson) but the route doesn't penetrate the neighbourhood's interior streets. Winnipeg has not built transit-oriented development (mixed residential + commercial anchors), so dense residential areas lack the ridership justification for more frequent routes — despite proximity to the PTN.

**Hypothesis:** Neighbourhoods with high population density and low transit stop density are the "TOD Gap" candidates — dense enough to warrant service, but structurally underserved due to pure-residential land use.

In [ ]:
# --- equity_combined.png: 2x3 panel (transit-oriented dev gap + transit access + equity) ---
import numpy as np
from matplotlib.colors import TwoSlopeNorm
from scipy.stats import zscore

neigh_gdf = ctx.working_db.neighbourhood_gdf()

# ── Data: Transit-Oriented Development gap ──
density_table = coverage_analyzer.neighbourhood_density()
equity_table = coverage_analyzer.equity_profile()
tod_df = None
if not density_table.empty and not equity_table.empty:
    tod_df = density_table.merge(
        equity_table[["neighbourhood_id", "population_total"]],
        on="neighbourhood_id", how="inner",
    )
    tod_df["pop_density"] = tod_df["population_total"] / tod_df["area_km2"].replace(0, float("nan"))
    tod_df = tod_df.dropna(subset=["pop_density", "stop_density_per_km2"])
    # Continuous gap score: high pop density + low stop density = positive gap
    tod_df["gap_score"] = (
        zscore(tod_df["pop_density"].fillna(0))
        - zscore(tod_df["stop_density_per_km2"].fillna(0))
    )

# ── Data: transit accessibility (r5py) ──
transit_matrix = ctx.working_db.query(
    "SELECT from_id, AVG(travel_time_p50) as avg_transit_min "
    "FROM ywg_transit_matrix_current WHERE from_id != to_id GROUP BY from_id"
)
walk_matrix = ctx.working_db.query(
    "SELECT from_id, AVG(travel_time_p50) as avg_walk_min "
    "FROM ywg_walk_matrix_current WHERE from_id != to_id GROUP BY from_id"
)
nb_summary = None
if not transit_matrix.empty and not walk_matrix.empty:
    access_df = transit_matrix.merge(walk_matrix, on="from_id", how="inner")
    access_df["transit_advantage_min"] = access_df["avg_walk_min"] - access_df["avg_transit_min"]
    access_df["da_uid"] = "461" + access_df["from_id"].astype(str).str.zfill(5)
    da_nb = ctx.working_db.query("""
        SELECT d.da_uid, n.name as neighbourhood
        FROM ywg_census_da d, ywg_neighbourhoods n
        WHERE ST_Intersects(ST_Centroid(d.geometry), n.geometry)
    """)
    nb_access = access_df.merge(da_nb, on="da_uid", how="inner")
    nb_summary = nb_access.groupby("neighbourhood").agg(
        mean_transit_min=("avg_transit_min", "mean"),
        mean_walk_min=("avg_walk_min", "mean"),
        mean_advantage=("transit_advantage_min", "mean"),
    ).reset_index()

# ── Data: equity dimensions ──
modal_share = coverage_analyzer.modal_share_by_neighbourhood()
census_nb = ctx.working_db.query(
    "SELECT neighbourhood, pct_recent_immigrants, median_household_income_2020 "
    "FROM ywg_census_by_neighbourhood"
)
jobs_access_comparison_table = coverage_analyzer.jobs_access_comparison(
    baseline_feed_id="2024-12-15"
)

# ── Build 2×3 figure ──
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

def _plot_choropleth(ax, gdf, col, cmap, label, title, norm=None):
    gdf.plot(column=col, ax=ax, legend=True, cmap=cmap, norm=norm,
             edgecolor="gray", linewidth=0.2, missing_kwds={"color": "lightgray"},
             legend_kwds={"label": label, "shrink": 0.5})
    cx.add_basemap(ax, source=cx.providers.CartoDB.Positron, zoom=12)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_axis_off()

# (a) Transit-Oriented Development gap — continuous choropleth
ax = axes[0, 0]
if tod_df is not None:
    gdf_tod = neigh_gdf.merge(
        tod_df[["neighbourhood", "gap_score"]],
        on="neighbourhood", how="left",
    ).to_crs(epsg=3857)
    vmax = gdf_tod["gap_score"].abs().quantile(0.95)
    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
    _plot_choropleth(ax, gdf_tod, "gap_score", "RdYlGn_r",
                     "Gap Score (red = underserved)",
                     "Transit-Oriented Development Gap", norm=norm)
else:
    ax.text(0.5, 0.5, "No data", ha="center", va="center"); ax.set_axis_off()

# (b) Average transit travel time
ax = axes[0, 1]
if nb_summary is not None:
    gdf_t = neigh_gdf.merge(nb_summary, on="neighbourhood", how="left").to_crs(epsg=3857)
    _plot_choropleth(ax, gdf_t, "mean_transit_min", "YlOrRd_r",
                     "Avg Transit Time (min)", "Transit Travel Time")
else:
    ax.text(0.5, 0.5, "No data", ha="center", va="center"); ax.set_axis_off()

# (c) Transit advantage over walking
ax = axes[0, 2]
if nb_summary is not None:
    vmax = nb_summary["mean_advantage"].abs().quantile(0.95)
    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
    _plot_choropleth(ax, gdf_t, "mean_advantage", "RdYlGn",
                     "Min Saved vs Walking", "Transit Advantage")
else:
    ax.text(0.5, 0.5, "No data", ha="center", va="center"); ax.set_axis_off()

# (d) Transit commute modal share
ax = axes[1, 0]
if not modal_share.empty:
    gdf1 = neigh_gdf.merge(modal_share, on="neighbourhood", how="left").to_crs(epsg=3857)
    col = [c for c in gdf1.columns if 'transit' in c.lower() or 'modal' in c.lower() or 'pct_commute' in c.lower()]
    plot_col = col[0] if col else modal_share.columns[-1]
    _plot_choropleth(ax, gdf1, plot_col, "YlOrRd",
                     "% Transit Commuters", "Transit Commute Share")
else:
    ax.text(0.5, 0.5, "No data", ha="center", va="center"); ax.set_axis_off()

# (e) Recent immigrant concentration
ax = axes[1, 1]
if not census_nb.empty:
    gdf2 = neigh_gdf.merge(census_nb, on="neighbourhood", how="left").to_crs(epsg=3857)
    _plot_choropleth(ax, gdf2, "pct_recent_immigrants", "GnBu",
                     "% Recent Immigrants", "Immigrant Concentration")
else:
    ax.text(0.5, 0.5, "No data", ha="center", va="center"); ax.set_axis_off()

# (f) Employment access change
ax = axes[1, 2]
if not jobs_access_comparison_table.empty:
    gdf3 = neigh_gdf.merge(
        jobs_access_comparison_table[["neighbourhood", "jobs_access_change"]],
        on="neighbourhood", how="left",
    ).to_crs(epsg=3857)
    vmax = max(gdf3["jobs_access_change"].abs().quantile(0.95), 10)
    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
    _plot_choropleth(ax, gdf3, "jobs_access_change", "RdYlGn",
                     "Jobs Access Change", "Employment Access Change", norm=norm)
else:
    ax.text(0.5, 0.5, "No data", ha="center", va="center"); ax.set_axis_off()

plt.tight_layout()
save_report_figure(fig, "equity_combined.png",
                   figures_dir=figure_output_directory, dpi=dpi, enabled=save_figures)
plt.close(fig)

In [ ]:
# (transit_accessibility consolidated into equity_combined above)